# Small language models on grade-school maths: Qwen3-1.7B vs SmolLM3-3B vs Phi-4-mini

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CJosh88/slms/blob/main/experiments/2026-09-slm-gsm8k/notebook.ipynb)

**Question:** Which popular small language model (4B parameters or fewer) solves grade-school maths word problems most accurately, and what does that accuracy cost in speed and memory?

**Setup:** 200 randomly chosen GSM8K test questions, one shared zero-shot step-by-step prompt, greedy decoding, thinking mode off, on a single Colab GPU (a free T4 is enough).

**TL;DR:** TODO(you) after running.

## Why this matters

Small language models are cheap to run locally, on a single consumer GPU, or at the edge, so they suit many product features where a frontier API would be overkill. Multi-step arithmetic reasoning is a common weak spot for small models, and GSM8K is a quick, well-known way to probe it. This notebook compares three freely downloadable instruct models of different sizes and origins under identical conditions: **Qwen3-1.7B** (Alibaba Qwen, Apache-2.0), **SmolLM3-3B** (Hugging Face, Apache-2.0) and **Phi-4-mini-instruct** (Microsoft, 3.8B, MIT). Qwen3 and SmolLM3 both have an optional "thinking" mode. It is switched off here so that all three models answer in the same style.

## Setup

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or better) before running. The versions below were pinned when the notebook was written. Colab already includes `torch`, `pandas` and `matplotlib`.

In [ ]:
%pip install -q "transformers==4.57.3" "accelerate==1.11.0" "datasets==4.8.5"

In [ ]:
import gc
import json
import math
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    # T4 (compute capability 7.5) has no native bf16, so use fp16 there.
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
else:
    DTYPE = torch.float32
print("dtype:", DTYPE)

## Config

Everything you can adjust is in this cell. Model outputs are cached in `CACHE_DIR` as one JSONL file per model, so a rerun (or a run interrupted by a Colab disconnect) skips questions that are already done. If you set `USE_DRIVE = True`, the cache is saved to Google Drive and survives the end of the Colab session.

In [ ]:
DATASET_ID = "openai/gsm8k"
DATASET_CONFIG = "main"
SPLIT = "test"
N_SAMPLES = 200          # 1,319 test questions in total; 200 keeps a T4 run to about 30 minutes

MAX_NEW_TOKENS = 512     # room for step-by-step reasoning
BATCH_SIZE = 8           # lower this if you hit CUDA out-of-memory errors

# chat_kwargs are passed to apply_chat_template. enable_thinking=False turns off
# the reasoning mode of Qwen3 and SmolLM3. Phi's template ignores it.
MODELS = {
    "Qwen3-1.7B": {"id": "Qwen/Qwen3-1.7B", "chat_kwargs": {"enable_thinking": False}},
    "SmolLM3-3B": {"id": "HuggingFaceTB/SmolLM3-3B", "chat_kwargs": {"enable_thinking": False}},
    "Phi-4-mini": {"id": "microsoft/Phi-4-mini-instruct", "chat_kwargs": {}},
}

SYSTEM_PROMPT = "You are a careful math tutor."
USER_TEMPLATE = (
    "Solve the following problem step by step. "
    "At the end, write the final answer as a single number on its own line in the form '#### <number>'.\n\n"
    "Problem: {question}"
)

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/slms/2026-09-slm-gsm8k")
else:
    BASE_DIR = Path(".")
CACHE_DIR = BASE_DIR / "cache"
FIG_DIR = BASE_DIR / "figures"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Data

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) (Cobbe et al., 2021) contains about 8.5K grade-school maths word problems that each take 2–8 steps to solve. The `main` config has 7,473 training and 1,319 test questions. Each reference solution ends with `#### <answer>`.

**Licence:** MIT.

In [ ]:
raw = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
print(raw)

rng = np.random.default_rng(SEED)
sample_idx = sorted(rng.choice(len(raw), size=N_SAMPLES, replace=False).tolist())


def parse_gold(answer: str) -> float:
    return float(answer.split("####")[-1].strip().replace(",", ""))


df = pd.DataFrame({
    "idx": sample_idx,
    "question": [raw[i]["question"] for i in sample_idx],
    "gold_solution": [raw[i]["answer"] for i in sample_idx],
})
df["gold"] = df["gold_solution"].map(parse_gold)
print(df.shape)
for _, row in df.head(3).iterrows():
    print(f"\n[{row.idx}] {row.question}\n-> gold: {row.gold}")

## Shared inference and answer extraction

This section plays the role of the "baseline" and "new approach" steps in other experiments. Here the only thing that changes between runs is the model. The prompt, decoding settings, batch size, maximum tokens and answer parser are all the same for every model.

- **Decoding:** greedy (`do_sample=False`), so results are deterministic.
- **Latency:** each batch is timed and the time is split evenly across the questions in it, giving seconds per question at `BATCH_SIZE`. It is not single-request latency.
- **Answer extraction:** the last `#### <number>` in the output. If that is missing, the last `\boxed{}` value. If that is missing too, the last number in the text. The notebook also reports how often the model used the requested format.

In [ ]:
NUM = r"-?\d[\d,]*(?:\.\d+)?"


def build_messages(question: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(question=question)},
    ]


def extract_pred(text: str):
    # Returns (value or None, how it was found).
    for pattern, how in [(rf"####\s*\$?\s*({NUM})", "####"), (rf"\\boxed\{{\s*\$?\s*({NUM})", "boxed")]:
        hits = re.findall(pattern, text)
        if hits:
            return float(hits[-1].replace(",", "")), how
    hits = re.findall(NUM, text)
    if hits:
        return float(hits[-1].replace(",", "")), "last_number"
    return None, "none"


def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open() as f:
        return {rec["idx"]: rec for rec in map(json.loads, f)}


def run_model(name: str, cfg: dict, data: pd.DataFrame) -> pd.DataFrame:
    cache_path = CACHE_DIR / f"{name}.jsonl"
    meta_path = CACHE_DIR / f"{name}_meta.json"
    done = load_cache(cache_path)
    todo = data[~data["idx"].isin(done)]
    print(f"{name}: {len(done)} cached, {len(todo)} to generate")

    if len(todo):
        tok = AutoTokenizer.from_pretrained(cfg["id"])
        tok.padding_side = "left"
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model = AutoModelForCausalLM.from_pretrained(cfg["id"], dtype=DTYPE, device_map="auto")
        model.eval()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        with cache_path.open("a") as f:
            for start in range(0, len(todo), BATCH_SIZE):
                batch = todo.iloc[start:start + BATCH_SIZE]
                prompts = [
                    tok.apply_chat_template(build_messages(q), tokenize=False,
                                            add_generation_prompt=True, **cfg["chat_kwargs"])
                    for q in batch["question"]
                ]
                enc = tok(prompts, return_tensors="pt", padding=True,
                          add_special_tokens=False).to(model.device)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                         pad_token_id=tok.pad_token_id)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                per_q = (time.perf_counter() - t0) / len(batch)

                gen = out[:, enc["input_ids"].shape[1]:]
                for idx, g in zip(batch["idx"], gen):
                    rec = {
                        "idx": int(idx),
                        "output": tok.decode(g, skip_special_tokens=True),
                        "new_tokens": int((g != tok.pad_token_id).sum()),
                        "sec_per_q": per_q,
                    }
                    f.write(json.dumps(rec) + "\n")
                    done[rec["idx"]] = rec
                print(f"  {min(start + BATCH_SIZE, len(todo))}/{len(todo)}", end="\r")

        peak_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else float("nan")
        meta_path.write_text(json.dumps({"peak_mem_gb": peak_gb, "dtype": str(DTYPE)}))
        del model, tok
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    res = pd.DataFrame([done[i] for i in data["idx"]])
    res["model"] = name
    return res

## Evaluation

The three models run one after another. Each model is loaded, run on the sample, and then removed from GPU memory before the next one loads.

In [ ]:
all_res = pd.concat([run_model(name, cfg, df) for name, cfg in MODELS.items()], ignore_index=True)
all_res = all_res.merge(df[["idx", "question", "gold"]], on="idx")
all_res[["pred", "extracted_by"]] = all_res["output"].apply(lambda t: pd.Series(extract_pred(t)))
all_res["correct"] = (all_res["pred"] - all_res["gold"]).abs() < 1e-6
all_res.to_csv(BASE_DIR / "results.csv", index=False)
all_res.head()

In [ ]:
def wilson(k, n, z=1.96):
    # 95% Wilson score interval for a proportion.
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


rows = []
for name, g in all_res.groupby("model", sort=False):
    meta_path = CACHE_DIR / f"{name}_meta.json"
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    lo, hi = wilson(g["correct"].sum(), len(g))
    rows.append({
        "model": name,
        "n": len(g),
        "accuracy": g["correct"].mean(),
        "ci_low": lo,
        "ci_high": hi,
        "used_####_format": (g["extracted_by"] == "####").mean(),
        "sec_per_q": g["sec_per_q"].mean(),
        "tokens_per_q": g["new_tokens"].mean(),
        "tokens_per_sec": g["new_tokens"].sum() / g["sec_per_q"].sum(),
        "hit_max_tokens": (g["new_tokens"] >= MAX_NEW_TOKENS).mean(),
        "peak_mem_gb": meta.get("peak_mem_gb", float("nan")),
    })
summary = pd.DataFrame(rows).set_index("model")
summary.to_csv(BASE_DIR / "summary.csv")
summary.style.format({
    "accuracy": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}", "used_####_format": "{:.0%}",
    "sec_per_q": "{:.2f}", "tokens_per_q": "{:.0f}", "tokens_per_sec": "{:.1f}",
    "hit_max_tokens": "{:.1%}", "peak_mem_gb": "{:.1f}",
})

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
err = [summary["accuracy"] - summary["ci_low"], summary["ci_high"] - summary["accuracy"]]
ax.bar(summary.index, summary["accuracy"], yerr=err, capsize=6, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_ylim(0, 1)
ax.set_ylabel("Exact-match accuracy")
ax.set_title(f"GSM8K accuracy (n={N_SAMPLES}, 95% Wilson CI)")
for i, v in enumerate(summary["accuracy"]):
    ax.text(i, v + 0.03, f"{v:.1%}", ha="center")

ax = axes[1]
for name, row in summary.iterrows():
    ax.scatter(row["sec_per_q"], row["accuracy"], s=max(row["peak_mem_gb"], 1) * 40, alpha=0.7)
    ax.annotate(f"{name}\n{row['peak_mem_gb']:.1f} GB", (row["sec_per_q"], row["accuracy"]),
                textcoords="offset points", xytext=(8, -4))
ax.set_xlabel(f"Seconds per question (batch size {BATCH_SIZE})")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs speed (bubble size = peak GPU memory)")

plt.tight_layout()
fig.savefig(FIG_DIR / "accuracy_and_speed.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([g["new_tokens"] for _, g in all_res.groupby("model", sort=False)],
           showfliers=False)
ax.set_xticks(range(1, len(MODELS) + 1), list(MODELS))
ax.set_ylabel("Generated tokens per question")
ax.set_title("How much does each model write?")
fig.savefig(FIG_DIR / "tokens_per_question.png", dpi=150, bbox_inches="tight")
plt.show()

wide = all_res.pivot(index="idx", columns="model", values="correct")[list(MODELS)]
print("Solved by all three:", int(wide.all(axis=1).sum()))
print("Solved by none:     ", int((~wide).all(axis=1).sum()))
print("Solved by exactly one model:")
print(wide[wide.sum(axis=1) == 1].idxmax(axis=1).value_counts())

## Qualitative examples

These are questions the models disagreed on, with at least one failure from each model where there is one. Read the reasoning, not just the final number. Was the error in the arithmetic, in reading the problem, or in the answer format?

In [ ]:
picked = []
for name in MODELS:  # at least one failure per model
    fails = wide.index[~wide[name] & wide.any(axis=1)].difference(picked)
    if len(fails):
        picked.append(fails[0])
disagree = wide.index[wide.any(axis=1) & ~wide.all(axis=1)].difference(picked)
picked += list(disagree[: max(0, 5 - len(picked))])

for idx in picked:
    q = df.set_index("idx").loc[idx]
    print("=" * 100)
    print(f"[{idx}] {q.question}\nGOLD: {q.gold}")
    for name in MODELS:
        r = all_res[(all_res.idx == idx) & (all_res.model == name)].iloc[0]
        mark = "OK " if r.correct else "BAD"
        tail = r.output.strip()[-400:].replace("\n", " ")
        print(f"\n  {mark} {name}: pred={r.pred}  (via {r.extracted_by})\n    ...{tail}")

## Interpretation

TODO(you): fill this in after running the full notebook.

- Which model won, and are the confidence intervals clearly separated or do they overlap?
- Is the most accurate model worth its extra time and memory?
- What kinds of errors did the qualitative examples show: arithmetic, misreading the problem, or answer format?
- What surprised you?
- When would you use each model, and when wouldn't you?

In [ ]:
# TODO(you): add any extra analysis here, e.g. accuracy by solution length (number of steps in gold_solution).

## Limitations and next steps

- **Sample size:** 200 questions gives confidence intervals of roughly ±6–7 points, so small gaps between models are not meaningful. Set `N_SAMPLES = 1319` to use the full test set.
- **One prompt, zero-shot, greedy decoding:** the model cards use different settings. Phi-4-mini's reported score uses 8-shot chain of thought, and Qwen recommends sampling rather than greedy decoding. Tuning the prompt for each model could change the ranking.
- **Thinking mode off:** Qwen3 and SmolLM3 would probably score higher with thinking on, at a large cost in tokens. That comparison is a natural follow-up experiment.
- **fp16 on T4:** these models were trained in bf16. Running them in fp16 on a T4 can occasionally cause numerical problems, so check outputs for gibberish.
- **Contamination:** GSM8K is public and widely used, so some test questions may have appeared in the models' training data.
- **Latency** is measured at batch size 8 on whatever GPU Colab assigned, so compare models only within the same run.

## Environment

In [ ]:
!pip freeze | grep -iE "^(torch|transformers|accelerate|datasets|pandas|matplotlib|numpy)=="
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv